In [1]:
import os

In [2]:
%pwd

'd:\\MLOPS\\Projects\\DataScienceProject1\\research'

In [3]:
os.chdir("../")

In [4]:
import pandas as pd

In [5]:
df = pd.read_csv("artifacts/data_ingestion/winequality-red.csv")

In [6]:
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [8]:
from dataclasses import dataclass
from pathlib import Path

In [9]:
@dataclass
class DataValidationConfig:
  root_dir: Path
  STATUS_FILE: str
  unzip_data_dir: Path
  all_schema: dict

In [10]:
from src.DataScienceProject.constants import *
from src.DataScienceProject.utils.common import read_yaml, create_directories

In [14]:
class ConfigurationManager:
  def __init__(self , config_filepath=CONFIG_FILE_PATH , params_filepath=PARAMS_FILE_PATH ,       schema_filepath=SCHEMA_FILE_PATH):
    self.config = read_yaml(config_filepath)
    self.params = read_yaml(params_filepath)
    self.schema = read_yaml(schema_filepath)
    
    create_directories([self.config.artifacts_root])
  
  def get_data_validation_config(self) -> DataValidationConfig:
    config = self.config.data_validation
    schema = self.schema.COLUMNS
    
    create_directories([config.root_dir])
    
    data_validation_config = DataValidationConfig(
      root_dir = config.root_dir,
      STATUS_FILE = config.STATUS_FILE,
      unzip_data_dir = config.unzip_data_dir,
      all_schema = schema
    )
    
    return data_validation_config


In [16]:
class DataValidation:
  def __init__(self , config:DataValidationConfig):
    self.config = config
  
  def validate_all_columns(self)->bool:
    try:
      validate_status = None
      data = pd.read_csv(self.config.unzip_data_dir)
      all_columns = list(data.columns)
      all_schema = self.config.all_schema
      for column in all_columns:
        if column not in all_schema:
          validate_status = False
          with open(self.config.STATUS_FILE , "w") as f:
            f.write(f"Validation status : {validate_status} ")
        else:
          validate_status = True
          with open(self.config.STATUS_FILE , "w") as f:
            f.write(f"Validation status : {validate_status} ")
      return validate_status
    except Exception as e:
      raise e

In [17]:
try:
  config = ConfigurationManager()
  data_validation_config = config.get_data_validation_config()
  data_validation = DataValidation(config=data_validation_config)
  status = data_validation.validate_all_columns()
  print(status)
except Exception as e:
  raise e

[2026-05-20 03:45:26,149]: INFO: common :yaml file: config\config.yaml loaded successfully 
[2026-05-20 03:45:26,151]: INFO: common :yaml file: params.yaml loaded successfully 
[2026-05-20 03:45:26,160]: INFO: common :yaml file: schema.yaml loaded successfully 
[2026-05-20 03:45:26,163]: INFO: common :created directory at: artifacts 
[2026-05-20 03:45:26,164]: INFO: common :created directory at: artifacts/data_validation 
True
